In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




Loading BokehJS ...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 14:55:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/16 14:55:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
path_to_release_folder = "../../data/25.06/"
path_to_intermediate_data_folder = "../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

all_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence")


efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)


g_p_s = session.spark.read.parquet(path_to_intermediate_data_folder + "genes_therapeutic_areas")
g_p_s.count()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/polina/Gentropy-manuscript/data/25.06/output/credible_set.

In [ ]:
v1 = session.spark.read.csv(
    path_to_intermediate_data_folder + "combined_evidence_for_constrain_plot.csv", header=True, inferSchema=True
)

In [ ]:
v1.show(1)

+---------------+--------+
|       targetId|  source|
+---------------+--------+
|ENSG00000121075|orphanet|
+---------------+--------+
only showing top 1 row



In [ ]:
v1.groupBy("source").count().show()

+--------------------+-----+
|              source|count|
+--------------------+-----+
|        all_diseases| 8285|
|          hc_paralog| 1587|
|    distant_ortholog|  830|
|       cancer_ChEMBL|  649|
|         gene_burden|  557|
|                omim| 4182|
|           inComplex| 3530|
|  cancer_driver_gene|  368|
|      withdrawn_drug|  166|
|   non_cancer_ChEMBL|  958|
|       selectiveGene|  930|
|      essential_gene| 1489|
|           gwas_eQTL| 3945|
|       gwas_with_pav| 1574|
|       liable_target|  214|
|            orphanet| 3814|
|              ChEMBL| 1137|
|          dd_related|  861|
|        pharmacogene|  538|
|trial_safety_concern|  440|
+--------------------+-----+
only showing top 20 rows



# FUSSIL


In [ ]:
target = session.spark.read.parquet(path_to_release_folder + "output/target")

In [8]:
# Create a list of valid chromosomes
valid_chromosomes = [str(i) for i in range(1, 23)] + ["X", "Y"]

In [ ]:
# target=session.spark.read.parquet("gs://open-targets-data-releases/25.06/output/target")
target = (
    target.filter(f.col("genomicLocation").getField("chromosome").isin(valid_chromosomes))
    .filter(f.col("biotype") == "protein_coding")
    .cache()
)
target.count()

20083

In [ ]:
target.printSchema()

root
 |-- id: string (nullable = true)
 |-- approvedSymbol: string (nullable = true)
 |-- biotype: string (nullable = true)
 |-- transcriptIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- canonicalTranscript: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: string (nullable = true)
 |-- canonicalExons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- genomicLocation: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: long (nullable = true)
 |    |-- end: long (nullable = true)
 |    |-- strand: integer (nullable = true)
 |-- alternativeGenes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- approvedName: string (nullable = true)
 |-- go: array (nullable = true)
 |    |-- element: struct (containsNull = tru

In [ ]:
target.show(2, truncate=False)

+---------------+--------------+--------------+--------------------------------------------------------------------+---------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------+----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
mouse_syn = (
    target.select("id", "homologues")
    .select("id", f.explode("homologues").alias("syn"))
    .filter(f.lower(f.col("syn.speciesName")) == "mouse")  # case-insensitive match
    .select("id", f.col("syn.targetGeneId").alias("mouse_symbol"))
    .na.drop(subset=["mouse_symbol"])  # drop null labels if any
    .distinct()  # unique pairs
)

In [ ]:
mouse_syn.show()

+---------------+------------------+
|             id|      mouse_symbol|
+---------------+------------------+
|ENSG00000128039|ENSMUSG00000029233|
|ENSG00000134758|ENSMUSG00000024317|
|ENSG00000139351|ENSMUSG00000096468|
|ENSG00000141540|ENSMUSG00000034714|
|ENSG00000149506|ENSMUSG00000024734|
|ENSG00000171962|ENSMUSG00000056598|
|ENSG00000178882|ENSMUSG00000037962|
|ENSG00000179119|ENSMUSG00000049516|
|ENSG00000180694|ENSMUSG00000043252|
|ENSG00000184697|ENSMUSG00000023906|
|ENSG00000204481|ENSMUSG00000066030|
|ENSG00000205133|ENSMUSG00000055963|
|ENSG00000205330|ENSMUSG00000095696|
|ENSG00000273173|ENSMUSG00000102627|
|ENSG00000274070|ENSMUSG00000015944|
|ENSG00000100416|ENSMUSG00000022386|
|ENSG00000124181|ENSMUSG00000016933|
|ENSG00000128699|ENSMUSG00000026097|
|ENSG00000140153|ENSMUSG00000037957|
|ENSG00000158234|ENSMUSG00000032463|
+---------------+------------------+
only showing top 20 rows



In [ ]:
mouse_syn.select("id").distinct().count()

17678

In [ ]:
mouse_syn.select("mouse_symbol").distinct().count()

18616

In [ ]:
mouse_syn.count()

22184

In [ ]:
mouse_syn.show(1)

+---------------+------------------+
|             id|      mouse_symbol|
+---------------+------------------+
|ENSG00000128039|ENSMUSG00000029233|
+---------------+------------------+
only showing top 1 row



## Fussil data


In [ ]:
fussil = session.spark.read.csv(
    "./SourceDataFile1_FUSIL_bins.txt",
    header=True,
    inferSchema=True,
    sep="\t",
)

In [ ]:
mapping = session.spark.read.csv(
    "./MGIBatchReport_20251120_112357.txt",
    header=True,
    inferSchema=True,
    sep="\t",
)

In [ ]:
mapping.show(1)

+-----------+----------+------------------+------+--------------------+-------------------+------------------+
|      Input|Input Type|MGI Gene/Marker ID|Symbol|                Name|       Feature Type|        Ensembl ID|
+-----------+----------+------------------+------+--------------------+-------------------+------------------+
|MGI:1098434|       MGI|       MGI:1098434|  Rgs5|regulator of G-pr...|protein coding gene|ENSMUSG00000026678|
+-----------+----------+------------------+------+--------------------+-------------------+------------------+
only showing top 1 row



In [ ]:
mapping.count()

4668

In [ ]:
fussil.show(1)

+----------+-----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+
|   HGNC_ID|     MGI_ID|IMPC_adult_viability|IMPC_embryo_viability| Avana_mean|Avana_mean_045|IMPC_Viability|adHOM_top_level_mp_name|adHEMI_top_level_mp_name|Viable_Phenotypes_updated|FUSIL_outliers|FUSIL|
+----------+-----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+
|HGNC:10001|MGI:1098434|              Viable|    InsuffData/NoData|0.020186795| Non-essential|        Viable|                      -|                       -|       Viable_nophenotype|            VN|   VN|
+----------+-----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+------------------

In [ ]:
fussil.groupBy("FUSIL").count().show()

+-----+-----+
|FUSIL|count|
+-----+-----+
|   VP| 1867|
|   SV|  421|
|   CL|  413|
|   DL|  764|
|    -|  881|
|   VN|  318|
+-----+-----+



In [ ]:
fussil.count()

4664

In [ ]:
mapping = mapping.select("input", "Ensembl ID").withColumnRenamed("input", "MGI_ID")

In [ ]:
fussil = fussil.join(mapping, on="MGI_ID", how="left")

In [ ]:
fussil.count()

4668

In [ ]:
fussil.show(1)

+-----------+----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+------------------+
|     MGI_ID|   HGNC_ID|IMPC_adult_viability|IMPC_embryo_viability| Avana_mean|Avana_mean_045|IMPC_Viability|adHOM_top_level_mp_name|adHEMI_top_level_mp_name|Viable_Phenotypes_updated|FUSIL_outliers|FUSIL|        Ensembl ID|
+-----------+----------+--------------------+---------------------+-----------+--------------+--------------+-----------------------+------------------------+-------------------------+--------------+-----+------------------+
|MGI:1098434|HGNC:10001|              Viable|    InsuffData/NoData|0.020186795| Non-essential|        Viable|                      -|                       -|       Viable_nophenotype|            VN|   VN|ENSMUSG00000026678|
+-----------+----------+--------------------+---------------------+-----------+--------------+------

In [ ]:
fussil.printSchema()

root
 |-- MGI_ID: string (nullable = true)
 |-- HGNC_ID: string (nullable = true)
 |-- IMPC_adult_viability: string (nullable = true)
 |-- IMPC_embryo_viability: string (nullable = true)
 |-- Avana_mean: string (nullable = true)
 |-- Avana_mean_045: string (nullable = true)
 |-- IMPC_Viability: string (nullable = true)
 |-- adHOM_top_level_mp_name: string (nullable = true)
 |-- adHEMI_top_level_mp_name: string (nullable = true)
 |-- Viable_Phenotypes_updated: string (nullable = true)
 |-- FUSIL_outliers: string (nullable = true)
 |-- FUSIL: string (nullable = true)
 |-- Ensembl ID: string (nullable = true)



In [ ]:
fussil = fussil.select("HGNC_ID", "FUSIL", "Ensembl ID").filter(~(f.col("FUSIL") == "-"))

In [ ]:
fussil.groupBy("FUSIL").count().show()

+-----+-----+
|FUSIL|count|
+-----+-----+
|   VP| 1869|
|   SV|  421|
|   CL|  413|
|   DL|  766|
|   VN|  318|
+-----+-----+



In [ ]:
fussil.count()

3787

In [ ]:
fussil.select("Ensembl ID").distinct().count()

3787

In [ ]:
# fussil = fussil.withColumn(
#    "hgnc_id_small",
#    f.lower(
#        f.regexp_replace(f.col("HGNC_ID").cast("string"), r"(?i)^hgnc:", "")
#    )
# )

In [ ]:
fussil = fussil.withColumnRenamed("Ensembl ID", "mouse_symbol")

In [ ]:
fussil.show()

+----------+-----+------------------+
|   HGNC_ID|FUSIL|      mouse_symbol|
+----------+-----+------------------+
|HGNC:10001|   VN|ENSMUSG00000026678|
|HGNC:10007|   DL|ENSMUSG00000025735|
|HGNC:10009|   VP|ENSMUSG00000028825|
|HGNC:10021|   VP|ENSMUSG00000022221|
|HGNC:10024|   VN|ENSMUSG00000039194|
| HGNC:1004|   DL|ENSMUSG00000029438|
|HGNC:10044|   VP|ENSMUSG00000035896|
|HGNC:10057|   VP|ENSMUSG00000036503|
|HGNC:10062|   CL|ENSMUSG00000028309|
|HGNC:10065|   VP|ENSMUSG00000045409|
|HGNC:10075|   CL|ENSMUSG00000009535|
| HGNC:1020|   CL|ENSMUSG00000026172|
|HGNC:10251|   SV|ENSMUSG00000024290|
|HGNC:10258|   SV|ENSMUSG00000032238|
|HGNC:10289|   CL|ENSMUSG00000000751|
|HGNC:10297|   DL|ENSMUSG00000053604|
|  HGNC:103|   DL|ENSMUSG00000064105|
| HGNC:1034|   DL|ENSMUSG00000035086|
| HGNC:1037|   VP|ENSMUSG00000090231|
|HGNC:10378|   CL|ENSMUSG00000039640|
+----------+-----+------------------+
only showing top 20 rows



In [ ]:
fussil.count()

3787

In [ ]:
fussil_j = fussil.join(mouse_syn, on="mouse_symbol", how="inner")

In [ ]:
fussil_j.count()

3803

In [ ]:
fussil_j.show(1)

+------------------+----------+-----+---------------+
|      mouse_symbol|   HGNC_ID|FUSIL|             id|
+------------------+----------+-----+---------------+
|ENSMUSG00000026678|HGNC:10001|   VN|ENSG00000143248|
+------------------+----------+-----+---------------+
only showing top 1 row



In [ ]:
fussil_j = fussil_j.select("FUSIL", "id").distinct()
fussil_j.count()

3802

In [ ]:
fussil_j.show()

+-----+---------------+
|FUSIL|             id|
+-----+---------------+
|   VP|ENSG00000144290|
|   VP|ENSG00000149635|
|   CL|ENSG00000117360|
|   DL|ENSG00000115289|
|   VP|ENSG00000124762|
|   DL|ENSG00000076043|
|   VN|ENSG00000204130|
|   VP|ENSG00000120675|
|   VP|ENSG00000204020|
|   VP|ENSG00000163472|
|   VP|ENSG00000247077|
|   VP|ENSG00000165660|
|   VP|ENSG00000131242|
|   VP|ENSG00000120341|
|   VP|ENSG00000136478|
|   VP|ENSG00000243364|
|   VP|ENSG00000131203|
|   VP|ENSG00000171747|
|   DL|ENSG00000241837|
|   CL|ENSG00000196235|
+-----+---------------+
only showing top 20 rows



In [48]:
fussil_j_prepared = (
    fussil_j.withColumn("source", f.concat(f.lit("fusil_"), f.col("FUSIL")))
    .withColumnRenamed("id", "targetId")
    .select("source", "targetId")
    .distinct()
)
fussil_j_prepared.show(5)

+--------+---------------+
|  source|       targetId|
+--------+---------------+
|fusil_VP|ENSG00000171206|
|fusil_CL|ENSG00000104671|
|fusil_VP|ENSG00000179431|
|fusil_VP|ENSG00000114737|
|fusil_CL|ENSG00000137812|
+--------+---------------+
only showing top 5 rows



# Constrain stratas


In [ ]:
target.show(1)

+---------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+----------------+-------------+--------------------+---------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+----------------+--------------------+--------------------+----+--------------------+--------------------+--------------+--------------------+--------------------+-----------------+--------+---------+
|             id|approvedSymbol|       biotype|       transcriptIds| canonicalTranscript|      canonicalExons|     genomicLocation|alternativeGenes| approvedName|                  go|hallmarks|            synonyms|      symbolSynonyms|        nameSynonyms|functionDescriptions|subcellularLocations|targetClass| obsoleteSymbols|       obsoleteNames|          constraint| tep|          proteinIds|             dbXrefs|chemicalProbes|          homologues|        tractability|safetyLiabilitie

In [ ]:
target_c = target.select("id", "biotype", "constraint").withColumnRenamed("id", "targetId")
target_c.show(5, truncate=False)

+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|targetId       |biotype       |constraint                                                                                                                                                                                               |
+---------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ENSG00000000003|protein_coding|[{syn, -0.47468, 30.657, 34, 1.1091, 0.843, 1.476, NULL, NULL, NULL}, {mis, 0.59223, 80.999, 66, 0.81482, 0.667, 1.0, NULL, NULL, NULL}, {lof, 0.066079, 7.865, 3, 0.38144, 0.173, 0.985, 10507, 5, 3}]  |
|ENSG00000000005|protein_coding|[{syn, -0.46371, 38.347, 42,

In [ ]:
target_c.count()

20083

In [ ]:
from pyspark.sql.functions import col, element_at, expr, size, when

target_with_constraints = (
    target_c.withColumn("syn_constr", expr("filter(constraint, x -> x.constraintType = 'syn')[0].score"))
    .withColumn("mis_constr", expr("filter(constraint, x -> x.constraintType = 'mis')[0].score"))
    .withColumn(
        "lof_constr",
        expr("filter(constraint, x -> x.constraintType = 'lof')[0].oeUpper"),
    )
)

In [ ]:
target_with_constraints.show(1)

+---------------+--------------+--------------------+----------+----------+----------+
|       targetId|       biotype|          constraint|syn_constr|mis_constr|lof_constr|
+---------------+--------------+--------------------+----------+----------+----------+
|ENSG00000000003|protein_coding|[{syn, -0.47468, ...|  -0.47468|   0.59223|     0.985|
+---------------+--------------+--------------------+----------+----------+----------+
only showing top 1 row



In [ ]:
target_with_constraints = target_with_constraints.withColumn("lof_constr_corrected", -f.col("lof_constr"))

In [ ]:
target_with_constraints.show(1)

+---------------+--------------+--------------------+----------+----------+----------+--------------------+
|       targetId|       biotype|          constraint|syn_constr|mis_constr|lof_constr|lof_constr_corrected|
+---------------+--------------+--------------------+----------+----------+----------+--------------------+
|ENSG00000000003|protein_coding|[{syn, -0.47468, ...|  -0.47468|   0.59223|     0.985|              -0.985|
+---------------+--------------+--------------------+----------+----------+----------+--------------------+
only showing top 1 row



In [ ]:
trq = target_with_constraints.dropna(subset=["lof_constr_corrected"])

# 2) compute quartile cutoffs and add quartile column (Q1..Q4)
q1, q2, q3 = trq.approxQuantile("lof_constr_corrected", [0.25, 0.5, 0.75], 0.0)

from pyspark.sql import functions as f

trq = trq.withColumn(
    "source",
    f.when(f.col("lof_constr_corrected") <= f.lit(q1), "lof_constr_Q1")
    .when(
        (f.col("lof_constr_corrected") > f.lit(q1)) & (f.col("lof_constr_corrected") <= f.lit(q2)),
        "lof_constr_Q2",
    )
    .when(
        (f.col("lof_constr_corrected") > f.lit(q2)) & (f.col("lof_constr_corrected") <= f.lit(q3)),
        "lof_constr_Q3",
    )
    .otherwise("lof_constr_Q4"),
).select("targetId", "source")

# quick check
trq.groupBy("source").count().show()

+-------------+-----+
|       source|count|
+-------------+-----+
|lof_constr_Q4| 4575|
|lof_constr_Q2| 4572|
|lof_constr_Q3| 4586|
|lof_constr_Q1| 4589|
+-------------+-----+



In [ ]:
trq.show(1)

+---------------+-------------+
|       targetId|       source|
+---------------+-------------+
|ENSG00000000003|lof_constr_Q2|
+---------------+-------------+
only showing top 1 row



# Combine all together


In [ ]:
v1.count()

57925

In [ ]:
v1.show(1)

+---------------+--------+
|       targetId|  source|
+---------------+--------+
|ENSG00000121075|orphanet|
+---------------+--------+
only showing top 1 row



In [ ]:
v2 = v1.unionByName(trq).unionByName(fussil_j_prepared)

In [ ]:
v2.count()

80049

In [ ]:
v2.groupBy("source").count().count()

32

In [ ]:
v1.groupBy("source").count().count()

23

In [ ]:
v2.show(1)

+---------------+--------+
|       targetId|  source|
+---------------+--------+
|ENSG00000121075|orphanet|
+---------------+--------+
only showing top 1 row



In [ ]:
v2.toPandas().to_csv(
    path_to_intermediate_data_folder + "list_of_genes_32_categories.csv",
    index=False,
)